## **BERT-CRF FOR NER**

source: 
- [article](https://stackoverflow.com/questions/79022870/how-do-i-add-a-crf-layer-to-a-bert-model-for-ner-tasks)
- [article](https://onlinelibrary.wiley.com/doi/full/10.1002/mgea.70001)
- [article](https://github.com/Louis-udm/NER-BERT-CRF/tree/master)
- [article](https://ceur-ws.org/Vol-4038/paper_33.pdf)

The error analysis of the fine-tuned BERT model show BIO boundary confusion, resulting in low F1-score. CRF is used to help enforce valid label transitions.

#### **MODEL ARCHITECTURE:**

- BERT + Linear Classifier + CRF

In [8]:
import sys
sys.path.append("../notebooks")
sys.path.append("..")  # Add project root so 'src' is importable


### **1. IMPORT DEPENDENCIES**

In [21]:
import importlib
import os
import csv
import time
from datetime import datetime

import torch
from datasets import load_from_disk

import src.bert_crf_model
importlib.reload(src.bert_crf_model)
from src.bert_crf_model import BertCRFForNER

import src.evaluation
importlib.reload(src.evaluation)
from src.evaluation import evaluate_resume_level_crf, save_to_dataframe

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)

import warnings
warnings.filterwarnings("ignore")


### **2. CONFIG**

In [10]:
MODEL_NAME = "bert-base-uncased"

# ==================== PATH ====================
DATASET_PATH = "../data/processed/resume_ner_hf"

OUT_DIR = f"../checkpoints/{MODEL_NAME}"
LOG_DIR = f"../logs/{MODEL_NAME}"
RESULT_CSV = "../results/experiment_results.csv"
ENTITY_CSV = "../results/per_entity_results.csv"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs("../results", exist_ok=True)

# ==================== MODEL HYPERPARAMETERS ====================
LEARNING_RATE = 2e-5        
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
EPOCH = 4           
WEIGHT_DECAY = 0.01
DROPOUT_RATE = 0.1          
WARMUP_RATIO = 0.1
STRIDE = 128        

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


### **3. LOAD DATASET**

In [12]:
ds = load_from_disk(DATASET_PATH)

# ==================== Label Setup ====================
# label should be manually define BIO tag
label_list = [
    "O",
    "B-JOB_TITLE",
    "I-JOB_TITLE",
    "B-SKILL",
    "I-SKILL",
    "B-EDUCATION",
    "I-EDUCATION",
]

id2label = {i: str(label) for i, label in enumerate(label_list)}
label2id = {str(label): i for i, label in enumerate(label_list)}

print("Data loaded successfully!")
print("Labels:", id2label)

Data loaded successfully!
Labels: {0: 'O', 1: 'B-JOB_TITLE', 2: 'I-JOB_TITLE', 3: 'B-SKILL', 4: 'I-SKILL', 5: 'B-EDUCATION', 6: 'I-EDUCATION'}


### **4. MODELLING**

In [16]:
model = BertCRFForNER(
    model_name=MODEL_NAME,
    num_labels=len(label_list),
    dropout_rate=DROPOUT_RATE
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=OUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCH,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_dir=LOG_DIR,
    logging_strategy="epoch",

    fp16=torch.cuda.is_available(),
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    data_collator=data_collator,
)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1635.45it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBO

### **5. TRAINING & VALIDATION**

In [17]:
start_time = time.time()
trainer.train()
end_time = time.time()

training_time = end_time - start_time

print(f"Training Time: {training_time/60:.2f} minutes")

Epoch,Training Loss,Validation Loss
1,98.938690,39.390404
2,37.642996,36.489925
3,31.003277,36.748852
4,26.813733,37.255867


Training Time: 98.45 minutes


In [22]:
val_results = evaluate_resume_level_crf(
    model=model,
    dataset=ds["validation"],
    id2label=id2label,
    data_collator=data_collator,
    device=device,
    stride=STRIDE,
    batch_size=EVAL_BATCH_SIZE
)

print("\n===== Resume-Level Validation Results =====")
print(f"Precision: {val_results['precision']:.4f}")
print(f"Recall:    {val_results['recall']:.4f}")
print(f"F1:        {val_results['f1']:.4f}")


===== Resume-Level Validation Results =====
Precision: 0.5985
Recall:    0.5469
F1:        0.5715


In [23]:
val_df = save_to_dataframe(
    results=val_results,
    model_name=MODEL_NAME,
    split="validation",
    file_path=ENTITY_CSV
)

display(val_df)

,timestamp,model,split,entity,precision,recall,f1,support
0,2026-05-16 18:01:15,bert-base-uncased,validation,EDUCATION,0.302083,0.432836,0.355828,201
1,2026-05-16 18:01:15,bert-base-uncased,validation,JOB_TITLE,0.683082,0.675570,0.679305,2546
2,2026-05-16 18:01:15,bert-base-uncased,validation,SKILL,0.585073,0.518401,0.549723,10706
3,2026-05-16 18:01:15,bert-base-uncased,validation,micro avg,0.598519,0.546867,0.571528,13453
4,2026-05-16 18:01:15,bert-base-uncased,validation,macro avg,0.523413,0.542269,0.528285,13453
5,2026-05-16 18:01:15,bert-base-uncased,validation,weighted avg,0.599393,0.546867,0.571349,13453


### **6. TESTING**

In [24]:
test_results = evaluate_resume_level_crf(
    model=model,
    dataset=ds["test"],
    id2label=id2label,
    data_collator=data_collator,
    device=device,
    stride=STRIDE,
    batch_size=EVAL_BATCH_SIZE
)

print("\n===== Resume-Level Test Results =====")
print(f"Precision: {test_results['precision']:.4f}")
print(f"Recall:    {test_results['recall']:.4f}")
print(f"F1:        {test_results['f1']:.4f}")


===== Resume-Level Test Results =====
Precision: 0.6136
Recall:    0.5663
F1:        0.5890


In [25]:
test_df = save_to_dataframe(
    results=val_results,
    model_name=MODEL_NAME,
    split="test",
    file_path=ENTITY_CSV
)

display(test_df)

,timestamp,model,split,entity,precision,recall,f1,support
0,2026-05-16 18:02:33,bert-base-uncased,test,EDUCATION,0.302083,0.432836,0.355828,201
1,2026-05-16 18:02:33,bert-base-uncased,test,JOB_TITLE,0.683082,0.675570,0.679305,2546
2,2026-05-16 18:02:33,bert-base-uncased,test,SKILL,0.585073,0.518401,0.549723,10706
3,2026-05-16 18:02:33,bert-base-uncased,test,micro avg,0.598519,0.546867,0.571528,13453
4,2026-05-16 18:02:33,bert-base-uncased,test,macro avg,0.523413,0.542269,0.528285,13453
5,2026-05-16 18:02:33,bert-base-uncased,test,weighted avg,0.599393,0.546867,0.571349,13453


### **7. LOG RESULTS**

In [26]:
def append_row_to_csv(file_path, row):
    file_exists = os.path.exists(file_path)

    with open(file_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())

        if not file_exists:
            writer.writeheader()

        writer.writerow(row)

if device == "cuda":
    max_gpu_mem = torch.cuda.max_memory_allocated(0) / 1e9
else:
    max_gpu_mem = 0

log_data = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model":MODEL_NAME,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "validation_precision": val_results["precision"],
    "validation_recall": val_results["recall"],
    "validation_f1": val_results["f1"],
    "test_precision": test_results["precision"],
    "test_recall": test_results["recall"],
    "test_f1": test_results["f1"],
    "training_time_min": round(training_time / 60, 2),
    "max_gpu_memory_gb": round(max_gpu_mem, 2),
    "device": device
}

append_row_to_csv(RESULT_CSV, log_data)

print(f"\nExperiment logged to: {RESULT_CSV}")
print(f"Per-entity results logged to: {ENTITY_CSV}")


Experiment logged to: ../results/experiment_results.csv
Per-entity results logged to: ../results/per_entity_results.csv


## Model Export

Export the trained BERT-CRF NER model for use on different machines.

**BERT-CRF is a custom architecture** — it cannot use `save_pretrained()` directly.
Export strategy:
- Save `model.state_dict()` with `torch.save()` for the full model weights
- Save the BERT config and tokenizer via HuggingFace for portability
- Save label mappings as JSON

**Loading on another machine:**
1. Install dependencies: `pip install transformers pytorch-crf`
2. Copy `src/bert_crf_model.py` alongside your script
3. Load as shown in the loading instructions printed below


In [27]:
import json
from pathlib import Path

EXPORT_DIR = Path("../exported_models/bert-crf-ner")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save full model state dict (custom architecture — use torch.save)
torch.save(model.state_dict(), EXPORT_DIR / "pytorch_model_state_dict.pt")

# Save tokenizer in HuggingFace format
tokenizer.save_pretrained(str(EXPORT_DIR))

# Save BERT base config for reconstruction
model.bert.config.save_pretrained(str(EXPORT_DIR))

# Save label mappings and model hyperparameters needed for reconstruction
export_config = {
    "id2label": id2label,
    "label2id": label2id,
    "label_list": label_list,
    "num_labels": len(label_list),
    "dropout_rate": DROPOUT_RATE,
    "base_model": MODEL_NAME,
    "task": "ner",
    "architecture": "BertCRFForNER"
}
with open(EXPORT_DIR / "label_config.json", "w") as f:
    json.dump(export_config, f, indent=2)

print(f"Model exported to: {EXPORT_DIR.resolve()}")
print(f"Files: {[f.name for f in EXPORT_DIR.iterdir()]}")
print()
print("=== Loading instructions ===")
print("# 1. Copy src/bert_crf_model.py to your project")
print("# 2. Install: pip install transformers pytorch-crf torch")
print("import torch, json")
print("from transformers import AutoTokenizer")
print("from bert_crf_model import BertCRFForNER  # from src/")
print(f'with open("{EXPORT_DIR}/label_config.json") as f: cfg = json.load(f)')
print('id2label = {int(k): v for k, v in cfg["id2label"].items()}')
print('label2id = cfg["label2id"]')
print('model = BertCRFForNER(num_labels=cfg["num_labels"], model_name=cfg["base_model"], dropout_rate=cfg["dropout_rate"])')
print(f'model.load_state_dict(torch.load("{EXPORT_DIR}/pytorch_model_state_dict.pt", map_location="cpu"))')
print('model.eval()')
print(f'tokenizer = AutoTokenizer.from_pretrained("{EXPORT_DIR}")')


Model exported to: /home/rzrdev/workspace/master/wqf7007-automated-cv-parser/exported_models/bert-crf-ner
Files: ['label_config.json', 'tokenizer_config.json', 'config.json', 'pytorch_model_state_dict.pt', 'tokenizer.json']

=== Loading instructions ===
# 1. Copy src/bert_crf_model.py to your project
# 2. Install: pip install transformers pytorch-crf torch
import torch, json
from transformers import AutoTokenizer
from bert_crf_model import BertCRFForNER  # from src/
with open("../exported_models/bert-crf-ner/label_config.json") as f: cfg = json.load(f)
id2label = {int(k): v for k, v in cfg["id2label"].items()}
label2id = cfg["label2id"]
model = BertCRFForNER(num_labels=cfg["num_labels"], model_name=cfg["base_model"], dropout_rate=cfg["dropout_rate"])
model.load_state_dict(torch.load("../exported_models/bert-crf-ner/pytorch_model_state_dict.pt", map_location="cpu"))
model.eval()
tokenizer = AutoTokenizer.from_pretrained("../exported_models/bert-crf-ner")
